In [ ]:
import random
import time
import os
import threading

# GESTIÓN DE MÚSICA DE FONDO

def reproducir_musica_fondo():
    """Reproduce un archivo de audio en bucle de forma nativa sin requerir pip externos"""
    import platform
    sistema = platform.system()
    
    # Nombre del archivo de audio en la misma carpeta
    archivo_audio = "pirate_theme.wav"
    
    if not os.path.exists(archivo_audio):
        return  # Si el archivo no existe, no hace nada para evitar errores

    try:
        if sistema == "Windows":
            import winsound
            while True:
                winsound.PlaySound(archivo_audio, winsound.SND_FILENAME | winsound.SND_LOOP | winsound.SND_ASYNC)
                time.sleep(60)
        elif sistema == "Darwin": # macOS
            while True:
                os.system(f"afplay '{archivo_audio}'")
        else: # Linux y otros
            while True:
                os.system(f"aplay '{archivo_audio}' >/dev/null 2>&1 || ffplay -nodisp -autoexit '{archivo_audio}' >/dev/null 2>&1")
    except Exception:
        pass

# Iniciar música en segundo plano en un hilo independiente
hilo_musica = threading.Thread(target=reproducir_musica_fondo, daemon=True)
hilo_musica.start()

# ESTILOS Y COLORES ANSI PARA TERMINAL

C_GOLD = "\033[93m"
C_RED = "\033[91m"
C_GREEN = "\033[92m"
C_CYAN = "\033[96m"
C_BLUE = "\033[94m"
C_BOLD = "\033[1m"
C_RESET = "\033[0m"

BANNER_PIRATA = f"""{C_GOLD}
          ⚓  -----------------------------------------  ⚓
             🏴‍☠️ ¡LA LEYENDA DEL TESORO DE PERDICIÓN! 🏴‍☠️
          ⚓  -----------------------------------------  ⚓{C_RESET}
"""

SKULL_ART = f"""{C_RED}
                   ☠️  ¡BOTÍN RECLAMADO! ☠️
                     _
                  .-"   "-.
                 /  _   _  \\
                |  (o) (o)  |
                |     <     |
                 \\  `---'  /
                  `------'
{C_RESET}"""



# FUNCIÓN: Crear y mezclar el tablero

def crear_tablero():
    cartas = [
        "🪙", "🪙",
        "💎", "💎",
        "⚓", "⚓",
        "🗝️", "🗝️",
        "🏴‍☠️", "🏴‍☠️",
        "💣", "💣",
        "🧭", "🧭",
        "🦜", "🦜"
    ]

    random.shuffle(cartas)

    tablero = []
    posicion = 0

    for fila in range(4):
        fila_tablero = []
        for columna in range(4):
            fila_tablero.append(cartas[posicion])
            posicion += 1
        tablero.append(fila_tablero)

    return tablero


# FUNCIÓN: Mostrar el tablero estilizado

def mostrar_tablero(tablero, descubiertas):
    print(f"\n{C_CYAN}         📜 MAPA DEL TESORO OCULTO 🗺️{C_RESET}")
    print(f"{C_GOLD}        1     2     3     4{C_RESET}")
    print(f"{C_BLUE}     +-----+-----+-----+-----+{C_RESET}")

    for fila in range(4):
        print(f"{C_GOLD}  {fila + 1}  {C_BLUE}|{C_RESET}", end="")

        for columna in range(4):
            if descubiertas[fila][columna]:
                print(f" {tablero[fila][columna]:^3} {C_BLUE}|{C_RESET}", end="")
            else:
                print(f"  📜 {C_BLUE}|{C_RESET}", end="")

        print()
        print(f"{C_BLUE}     +-----+-----+-----+-----+{C_RESET}")


# FUNCIÓN: Verificar posición

def pedir_posicion(descubiertas, numero):
    while True:
        print(f"\n{C_BOLD}🧭 Selecciona la coordenada {numero}:{C_RESET}")

        try:
            fila = int(input(f"{C_CYAN}  Fila (1-4): {C_RESET}"))
            columna = int(input(f"{C_CYAN}  Columna (1-4): {C_RESET}"))
        except ValueError:
            print(f"{C_RED}☠️ ¡Por la barba de Barbanegra! Ingresa números válidos.{C_RESET}")
            continue

        if fila < 1 or fila > 4 or columna < 1 or columna > 4:
            print(f"{C_RED}☠️ Te saliste del mapa, grumete (Elige de 1 a 4).{C_RESET}")
            continue

        fila -= 1
        columna -= 1

        if descubiertas[fila][columna]:
            print(f"{C_RED}☠️ ¡Ese cofre ya fue desenterrado y reclamado!{C_RESET}")
            continue

        return fila, columna


# FUNCIÓN: Turno del jugador

def turno_jugador(tablero, descubiertas, puntos_jugador):
    print(f"\n{C_GOLD}========================================{C_RESET}")
    print(f"{C_BOLD}   ⚔️ TURNO DEL CAPITÁN (JUGADOR){C_RESET}")
    print(f"{C_GOLD}========================================{C_RESET}")

    mostrar_tablero(tablero, descubiertas)

    fila1, columna1 = pedir_posicion(descubiertas, 1)
    descubiertas[fila1][columna1] = True
    mostrar_tablero(tablero, descubiertas)

    while True:
        fila2, columna2 = pedir_posicion(descubiertas, 2)
        if fila1 == fila2 and columna1 == columna2:
            print(f"{C_RED}☠️ ¡No puedes elegir el mismo cofre dos veces!{C_RESET}")
        else:
            break

    descubiertas[fila2][columna2] = True
    mostrar_tablero(tablero, descubiertas)

    time.sleep(0.8)

    if tablero[fila1][columna1] == tablero[fila2][columna2]:
        print(f"\n{C_GREEN}💰 ¡ARRR! ¡ENCONTRASTE UN BOTÍN COINCIDENTE! (+4 Doblones){C_RESET}")
        puntos_jugador += 4
    else:
        print(f"\n{C_RED}💥 ¡POR LAS BARBAS DE NEPTUNO! No coinciden. Las marea vuelve a cubrir las cartas. (-1 Doblón){C_RESET}")
        puntos_jugador -= 1
        descubiertas[fila1][columna1] = False
        descubiertas[fila2][columna2] = False

    print(f"\n👤 Doblones de tu tripulación: {C_GOLD}{puntos_jugador}{C_RESET}")
    input(f"\n{C_CYAN}Presiona ENTER para pasar el catalejo...{C_RESET}")
    return puntos_jugador


# FUNCIÓN: Turno de la máquina (Capitán Barbanegra)
def posiciones_disponibles(descubiertas):
    posiciones = []
    for fila in range(4):
        for columna in range(4):
            if not descubiertas[fila][columna]:
                posiciones.append((fila, columna))
    return posiciones


def turno_maquina(tablero, descubiertas, puntos_maquina):
    print(f"\n{C_RED}========================================{C_RESET}")
    print(f"{C_BOLD}   🏴‍☠️ TURNO DEL CAPITÁN BARBANEGRA (IA){C_RESET}")
    print(f"{C_RED}========================================{C_RESET}")

    posiciones = posiciones_disponibles(descubiertas)
    posicion1 = random.choice(posiciones)
    posiciones.remove(posicion1)
    posicion2 = random.choice(posiciones)

    fila1, columna1 = posicion1
    fila2, columna2 = posicion2

    print(f"\n{C_CYAN}🦜 Barbanegra está consultando sus cartas navegantes...{C_RESET}")
    time.sleep(1.2)
    print(f"⚔️ Revela cofre en ({fila1 + 1}, {columna1 + 1}) y cofre en ({fila2 + 1}, {columna2 + 1})")

    descubiertas[fila1][columna1] = True
    descubiertas[fila2][columna2] = True
    mostrar_tablero(tablero, descubiertas)

    time.sleep(1.2)

    if tablero[fila1][columna1] == tablero[fila2][columna2]:
        print(f"\n{C_RED}💀 ¡MALDICIÓN! Barbanegra ha saqueado una pareja. (+4 Doblones rival){C_RESET}")
        puntos_maquina += 4
    else:
        print(f"\n{C_GREEN}🌊 ¡Barbanegra falló su excavación! (-1 Doblón rival){C_RESET}")
        puntos_maquina -= 1
        descubiertas[fila1][columna1] = False
        descubiertas[fila2][columna2] = False

    print(f"\n🤖 Botín de Barbanegra: {C_GOLD}{puntos_maquina}{C_RESET}")
    input(f"\n{C_CYAN}Presiona ENTER para pasar el mando...{C_RESET}")
    return puntos_maquina


# CICLO PRINCIPAL

def juego_terminado(descubiertas):
    for fila in range(4):
        for columna in range(4):
            if not descubiertas[fila][columna]:
                return False
    return True


def mostrar_resultado(puntos_jugador, puntos_maquina):
    print(SKULL_ART)
    print(f"{C_GOLD}👤 Tu Puntuación: {puntos_jugador} doblones{C_RESET}")
    print(f"{C_RED}🤖 Puntuación de Barbanegra: {puntos_maquina} doblones{C_RESET}\n")

    if puntos_jugador > puntos_maquina:
        print(f"{C_GREEN}🏆 ¡VICTORIA PIRATA! Has dominado los siete mares y reclamado el tesoro.{C_RESET}")
    elif puntos_maquina > puntos_jugador:
        print(f"{C_RED}☠️ ¡DERROTA! Barbanegra te ha enviado a alimentarte con los peces.{C_RESET}")
    else:
        print(f"{C_CYAN}🤝 ¡EMPATE EN ALTA MAR! Se repartirán el botín equitativamente.{C_RESET}")


def jugar():
    tablero = crear_tablero()
    descubiertas = [[False] * 4 for _ in range(4)]
    puntos_jugador = 0
    puntos_maquina = 0

    print(BANNER_PIRATA)
    print("Encuentra las 8 parejas de objetos piratas antes que tu rival.")
    input(f"\n{C_GOLD}Presiona ENTER para desplegar velas...{C_RESET}")

    while not juego_terminado(descubiertas):
        puntos_jugador = turno_jugador(tablero, descubiertas, puntos_jugador)
        if juego_terminado(descubiertas):
            break

        puntos_maquina = turno_maquina(tablero, descubiertas, puntos_maquina)

    for fila in range(4):
        for columna in range(4):
            descubiertas[fila][columna] = True

    mostrar_tablero(tablero, descubiertas)
    mostrar_resultado(puntos_jugador, puntos_maquina)


# Inicio
if _name_ == "_main_":
    while True:
        jugar()
        print(f"\n{C_GOLD}========================================{C_RESET}")
        print("1. 🗺️ Iniciar una nueva expedición")
        print("2. ⚓ Retirarse al puerto (Salir)")
        opcion = input(f"\n{C_CYAN}Selecciona (1-2): {C_RESET}")

        if opcion == "1":
            continue
        else:
            print(f"\n{C_GREEN}⚓ ¡Levanten anclas y hasta la próxima travesía!{C_RESET}")
            break

: 